# 📌 Confidence Scoring
![Topic](https://img.shields.io/badge/Confidence Scoring%20Me-blue?style=flat-square)
![Category](https://img.shields.io/badge/Guardrails-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-June%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — To prevent an AI from making critical mistakes, you can set up a "Human-in-the-Loop" fallback. Instead of letting the AI guess blindly or blindly trusting its own text self-evaluation, you can extract its raw mathematical confidence scores (logprobs) directly from the API. If the confidence falls below 95%, the system automatically blocks the task and routes it to a human. </span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install openai` |
| Concepts | List any background knowledge needed |

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

When an AI handles sensitive operations like executing database queries, processing financial transactions, or calling external APIs (Tools / Function Calling), you cannot afford hallucinations. 

Beginners often mistakenly ask the AI "Are you sure? Rate your confidence from 1 to 100", but **LLMs are naturally overconfident and will lie about being sure.** 

**The only reliable approach is to inspect the model's actual token probabilities (logprobs) before allowing code to execute.**

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

### 2.1 Token Prediction

LLMs generate text token by token (word pieces). For every single token, the model calculates a probability distribution across its entire vocabulary.

### 2.2 Logprobs

The API converts these percentages into logarithmic probabilities (where $100$% = $0$, and lower probabilities become more negative). A 95% confidence threshold translates mathematically to a logprob of roughly -0.051 ($e^{-0.051} \approx 0.95$).

### 2.2 Min or Mean

**Min:** Scans every token in the generated output and blocks execution if any single one drops below the 95% confidence threshold. This is the most conservative approach. It is useful when every token is equally critical (e.g. a medical dosage, an account number). The drawback is that rare but harmless tokens like punctuation or JSON syntax can trigger a false block even when the model was confident about the meaningful content.

**Mean:**  Averages the confidence scores across all generated tokens and blocks execution if the result drops below 95%. This is more robust in practice, it isolated low-confidence tokens (formatting, quotes) get smoothed out by the surrounding high-confidence ones. The tradeoff is that a critically uncertain token (like an ambiguous user ID) can be masked by a sea of confident filler tokens.

---
## 3. Code Example

> **Goal:** Here is how to safely secure an OpenAI tool call using logprobs in Python.

In [ ]:
import math
import json
from openai import OpenAI

client = OpenAI()

user_prompt = "Delete the account for user... uh, I think it's usr_9948, or maybe 9949?"

# Ask for a structured JSON response in the text content
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. When the user wants to delete an account, "
                "respond ONLY with a JSON object like: "
                '{"action": "delete_user_account", "user_id": "<id>"} '
                "If you are unsure about the user_id, set user_id to null."
            )
        },
        {"role": "user", "content": user_prompt}
    ],
    logprobs=True
)

message = response.choices[0].message
content_logprobs = response.choices[0].logprobs.content

# Check if logprobs are available
if not content_logprobs:
    print("No logprobs available. Blocking for security.")
else:
    #  Mean of logprobs 
    avg_logprob = sum(t.logprob for t in content_logprobs) / len(content_logprobs)
    avg_confidence = math.exp(avg_logprob) * 100

    threshold = math.log(0.95)

    if avg_logprob >= threshold:
        print(f"Sufficient Confidence ({avg_confidence:.2f}%). Executing the tool.")
        parsed = json.loads(message.content)

        if parsed.get("user_id") is None:
            print("The AI is uncertain about the user_id. Routing to human agent.")
        else:
            print(f"Deleting account : {parsed['user_id']}")
            # delete_user_account(parsed["user_id"])
    else:
        print(f"Insufficient Confidence ({avg_confidence:.2f}%). Routing to human agent.")

---
## 4. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **LLMs are inherently overconfident self-evaluators.**
- **Logprobs represent true underlying math.**
- **The "Weakest Link" method provides maximum security.**
- **Tool calls generate JSON strings.**

</div>